In [1]:
import matplotlib
%matplotlib tk

import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

from scipy import ndimage as ndi
from skimage.segmentation import watershed
from skimage.feature import peak_local_max
import scipy
import os
import time

#
from utils import ComputeROIs

Autosaving every 180 seconds


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname = r"D:\User Training\Readtest8105\Image_001_001.raw"
fname = '/home/cat/data/donato/bscope_tests/image_10000frames.raw'
#fname = '/media/cat/4TB/donato/BSCOPE_tests/image_27000frames.raw'
# 
gr = ComputeROIs(fname)

#
gr.subsample = 3 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations


In [3]:
#######################################################################
########### COMPUTE STD OVER TIME TO GET CELL FOOTPRINTS ##############
#######################################################################
# TODO: the gaussian filter step takes a long time!!
# need to exponse some more variables here like imshow vmin/vmax which are used to plot the final result
start = time.time()
std_map = gr.make_std_map()
print ("total processing time: ", time.time()-start, " sec")

memmap :  (10000, 512, 512)
data into analysis:  (3334, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
staring computing std...
done computing std...
total processing time:  122.325350522995  sec


In [4]:
#######################################################################
########### RUN WATERSHED ALGORITHM DETECTION ON STD MAP ##############
#######################################################################
gr.find_roi_boundaries(std_map)

#
gr.scale=3000                      # spacing between each neuron trace because they are not normalized to the max vlaue
gr.trace_subsample = 5             # Subsample the time series to go faster;

# visualize traces
gr.compute_traces2()

memmap :  (10000, 512, 512)


100%|██████████████████████████████████████| 2000/2000 [00:03<00:00, 611.09it/s]


In [5]:
#####################################################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES AND SAVE DATA TO DISK #############
#####################################################################################

# save ensemble rois
gr.ensemble1 = [1,9]
gr.ensemble2 = [15,16]

# save all rois
np.savetxt(os.path.join(os.path.split(fname)[0],
                        'all_rois.txt'), np.int32(gr.rois), delimiter=',')

# Save pixel indexes for each cell, should be hopefully fast to broadcast live
both = np.hstack((gr.ensemble1, gr.ensemble2))
cells = []
for k in range(4):
    temp = gr.indexes[both[k]]
    temp1 = temp[0]
    temp2 = temp[1]
    temp = np.vstack((temp1,temp2))

    cells.append(temp)

# save individual pixels of each cell - currently implemented in BMI
np.savez(os.path.join(os.path.split(fname)[0],
                        'rois.npz'),
            cell0 = cells[0],
            cell1 = cells[1],
            cell2 = cells[2],
            cell3 = cells[3],
            cell_centres = np.int32(gr.rois)[both],
            cell_ids = both
        )

#
gr.show_traces_ids(both)


### COMPUTE THE MIN AND MAXES FOR THE SELECTED ENSMEMLES

Some important points:

1. For now we are working in pixel absolute values for each cell.  

2. A better option might be to find the maximum peak of a cell during a window and then save that value and normalize all future events by that value. (note: any online BMI filtering/chagnig of data will need to account for this).


In [10]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
# # NOT SURE THIS IS NECESSARY... TO CHECK 
gr.trace_subsample = 1             # Subsample the time series to go faster;

# visualize traces
gr.compute_traces2()


memmap :  (10000, 512, 512)


100%|████████████████████████████████████| 10000/10000 [00:13<00:00, 767.67it/s]


In [11]:
#####################################################################################
########### SELECT CELLS TO BE USED FOR ENSEMBLES AND SAVE DATA TO DISK #############
#####################################################################################

############################################
gr.sample_rate = 30
gr.post_reward_lockout = 10   # reward lockout in seconds
gr.balance_ensemble_rewards_flag = True   #this makes sure that both ensembles elicit a similar number of random rewards
gr.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
gr.smooth_diff_function = True    # use a kernel window to smooth current value

# find 30% reward threshold
gr.find_reward_thresholds()
print ("thresholds: ", gr.low, gr.high)

########################################
gr.plot_rewarded_ensembles()



difference between 2 ensembles:  (10000,)
nsec recording:  333
max number of rewards received randomly (i.e. every 30sec)  11
updated rwards #:  0 -3914.158836206897 4381.250192307692
updated rwards #:  0 -3718.450894396552 4162.187682692307
updated rwards #:  0 -3532.5283496767242 3954.0782985576916
updated rwards #:  0 -3355.9019321928877 3756.3743836298067
updated rwards #:  0 -3188.106835583243 3568.5556644483163
updated rwards #:  0 -3028.7014938040807 3390.1278812259
updated rwards #:  0 -2877.2664191138765 3220.6214871646052
updated rwards #:  0 -2733.4030981581827 3059.590412806375
updated rwards #:  0 -2596.7329432502734 2906.610892166056
updated rwards #:  0 -2466.8962960877598 2761.2803475577534
updated rwards #:  0 -2343.5514812833717 2623.2163301798655
updated rwards #:  0 -2226.373907219203 2492.055513670872
updated rwards #:  0 -2115.0552118582427 2367.4527379873284
updated rwards #:  0 -2009.3024512653305 2249.080101087962
updated rwards #:  0 -1908.837328702064 2136.62

In [ ]:
#######################################################
######### MANUAL ROI SELECTOR - DO NOT DELETE #########
#######################################################

# # importing the module
# import cv2

# # function to display the coordinates of
# # of the points clicked on the image
# def click_event(event, x, y, flags, params):

#     # checking for left mouse clicks
#     if event == cv2.EVENT_LBUTTONDOWN:

#         # displaying the coordinates
#         # on the Shell
#         print(x, ' ', y)
#         rois_manual.append([x,y])

#         # displaying the coordinates
#         # on the image window
#         #font = cv2.FONT_HERSHEY_SIMPLEX
#         img[y-2:y+3, x-2:x+3] = 0
   
#         #cv2.putText(img, str(x) + ',' +
#         #            str(y), (x,y), font,
#         #            .3, (255, 0, 0), 2)
#         cv2.imshow('image', img)

#     # checking for right mouse clicks	
#     #if event==cv2.EVENT_RBUTTONDOWN:
#     #    
#     #    np.save()

# # driver function
# if __name__=="__main__":
    
#     global rois_manual
    
#     rois_manual = []
    
#     # reading the image
#     #img = cv2.imread('lena.jpg', 1)
#     img = std_map.copy()
    
#     img = cv2.resize(img, (int(img.shape[0]*1.5),
#                            int(img.shape[1]*1.5))) 

#     # displaying the image
#     cv2.imshow('image', img)

#     # setting mouse handler for the image
#     # and calling the click_event() function
#     cv2.setMouseCallback('image', click_event)

#     # wait for a key to be pressed to exit
#     cv2.waitKey(0)

#     # close the window
#     cv2.destroyAllWindows()

# print (" DONE LABELING: ")
# print ("ROIS: ", rois_manual)

